In [1]:
# ================================
# IMPORTS
# ================================
import json
import re
import os
from collections import defaultdict


# ================================
# PATHS
# ================================
input_path = "data/processed/data.json"
output_path = "data/processed/cleaned_data.json"


# ================================
# LOAD DATA
# ================================
if not os.path.exists(input_path):
    raise FileNotFoundError(f"{input_path} not found. Run Notebook 01 first.")

with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} documents")


# ================================
# HELPER FUNCTIONS
# ================================

# Advanced cleaning
def normalize_text(text):
    if not text:
        return ""

    text = text.lower()
    text = re.sub(r'\s+', ' ', text)  # remove extra spaces
    text = re.sub(r'\b\d+\b', '', text)  # remove standalone numbers
    text = re.sub(r'[^a-zA-Z.,()%\- ]+', '', text)  # keep useful chars only

    return text.strip()


# Remove duplicate texts
def remove_duplicates(data):
    seen = set()
    unique_data = []

    for item in data:
        text = item.get("text", "").strip()

        if text and text not in seen:
            seen.add(text)
            unique_data.append(item)

    return unique_data


# Filter small / low-quality text
def filter_data(data, min_length=200):
    filtered = []

    for item in data:
        text = item.get("text", "")

        if len(text) >= min_length:
            filtered.append(item)

    return filtered


# Remove near-empty or noisy text
def remove_noise(data):
    cleaned = []

    for item in data:
        text = item.get("text", "")

        # remove if too many repeated chars
        if len(set(text)) < 10:
            continue

        cleaned.append(item)

    return cleaned


# ================================
# PIPELINE
# ================================

print("\n🔹 Removing duplicates...")
data = remove_duplicates(data)
print(f"After deduplication: {len(data)}")

print("\n🔹 Filtering small text...")
data = filter_data(data, min_length=200)
print(f"After length filter: {len(data)}")

print("\n🔹 Removing noisy entries...")
data = remove_noise(data)
print(f"After noise removal: {len(data)}")

print("\n🔹 Normalizing text...")

for item in data:
    item["text"] = normalize_text(item["text"])


# ================================
# DOMAIN DISTRIBUTION (for insight)
# ================================
domain_count = defaultdict(int)

for item in data:
    domain = item.get("domain", "unknown")
    domain_count[domain] += 1

print("\n📊 Domain Distribution:")
for k, v in domain_count.items():
    print(f"{k}: {v}")


# ================================
# SAVE CLEAN DATA
# ================================
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4)

print(f"\n✅ Cleaned data saved to {output_path}")


# ================================
# PREVIEW
# ================================
print("\n🔹 Sample Cleaned Data:\n")

for item in data[:2]:
    print(item)
    print("\n----------------------\n")

Loaded 26 documents

🔹 Removing duplicates...
After deduplication: 20

🔹 Filtering small text...
After length filter: 18

🔹 Removing noisy entries...
After noise removal: 18

🔹 Normalizing text...

📊 Domain Distribution:
pest: 1
fertilizer: 8
general: 2
weather: 2
soil: 5

✅ Cleaned data saved to data/processed/cleaned_data.json

🔹 Sample Cleaned Data:

{'text': 'an official website of the united states government the .gov means its official.federal government websites often end in .gov or .mil. before sharing sensitive information, make sure youre on a federal government site. the site is secure.the https ensures that you are connecting to the official website and that any information you provide is encrypted and transmitted securely. if you are giving a presentation about an environmental health topic or just looking for general information about environmental health research or the institute, this webpage will help. a resource for kids, parents, and teachers to find fun and educatio